In [1]:
import pandas as pd
import pickle
import numpy as np 
import random
from tqdm import tqdm
from pathlib import Path
import re

In [2]:
# file_out_dir = list(Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/').glob('*dataset*.pdpkl'))
# Import timit data - this df contains all the relevant signals, labels and data 
path = '/om2/user/imgriff/datasets/timit/all_timit_wsn_compatible.pdpkl'

timit_excerpts = pd.read_pickle(path)


In [5]:
## Normalize all audio signals 
all_signals = np.stack(timit_excerpts.signal)

all_signals = all_signals - all_signals.mean(1).reshape(-1,1) # de-mean rows 


per_signal_rms = np.sqrt(np.mean(np.power(all_signals, 2), axis=1)).reshape(-1,1) # per signal rms 


all_signals = all_signals * 0.1 / per_signal_rms # normalize all signals to rms of 0.02 or ~60dB

timit_excerpts['signal'] = [signal for signal in all_signals] # update signal column

In [13]:
# parse target & rest 

target_timit = timit_excerpts[timit_excerpts.word_int != 0]
# target_sentences = target_timit.sentence_id.unique()

# filter cues 
cue_timit = timit_excerpts[timit_excerpts.word_int == 0]
# cue_timit = timit_excerpts[(timit_excerpts.word_int == 0) & (~timit_excerpts.sentence_id.str.contains('1|2'))]

# cut targets to include talkers included in both cue and target lest 
# # Cut targets to minimize word repetition 
valid_words = target_timit.word.value_counts()[target_timit.word.value_counts() < 5].index

# get training stim for start of experiment 
training_timit = target_timit[~target_timit.word.isin(valid_words)] # can use words with more reps for training set

target_timit = target_timit[target_timit.word.isin(valid_words)]

valid_talkers = np.intersect1d(target_timit.speaker.unique(),cue_timit.speaker.unique(), assume_unique=True)
target_timit = target_timit[target_timit.speaker.isin(valid_talkers)]



In [21]:
next_pow_2 = lambda x: int(pow(2, np.ceil(np.log2(x))))

def combine_with_noise(clean, noise, snr):
    # get ratio in rms 
    rms_ratio = np.power(10.0, snr / 20.0)
    
    # remove DC of each signal
    clean = clean - clean.mean()
    noise = noise - noise.mean()
    # get rms of each signal
    clean_rms = np.sqrt(np.mean(np.power(clean, 2)))
    noise_rms = np.sqrt(np.mean(np.power(noise, 2)))
    # scale factor for setting noise to desired SNR 
    scale_factor = clean_rms / (noise_rms * rms_ratio)
    # Blend signals 
    noise = noise * scale_factor
    mixture = clean + noise[:len(clean)]
    return mixture, scale_factor

def rms_normalize(wav, new_rms=0.1, axis=0): 
    wav = wav - wav.mean(axis=axis)
    rms_wav = np.sqrt(np.mean(np.power(wav, 2), axis=axis))
    scale_factor = new_rms / rms_wav
    wav = wav * scale_factor
    return wav, scale_factor

## Define windowing function - will apply a cosine ramp to the start and end of a signal

In [22]:
## Precompute spectrum of all signals for speech shaped noise 

all_foregrounds = np.stack(target_timit['signal'])
n_x = all_foregrounds.shape[-1]
n_fft = next_pow_2(n_x)
X = np.fft.rfft(all_foregrounds, next_pow_2(n_fft))
X = X.mean(0)

def generate_speech_shaped_noise(X=X):
    # Randomize phase.
    noise_mag = np.abs(X) * np.exp(
        2 * np.pi * 1j * np.random.random(X.shape[-1]))
    noise = np.real(np.fft.irfft(noise_mag, n_fft))
    ssn = noise[:n_x]
    return ssn

In [36]:
## Make trial conditions 
import itertools


# setup design
snrs = [-6, -3, 0, 3] 

single_talker_conds = ['m', 'f']

two_talker_conds = list(itertools.combinations_with_replacement(distractor_types,2))
four_talker_conds = list(itertools.combinations_with_replacement(distractor_types,4))

distractor_types + two_talker_conds + four_talker_conds + ['ssn']

distractor_conds = single_talker_conds + two_talker_conds + four_talker_conds + ['ssn']
condition_pairs = list(itertools.product(snrs, distractor_conds))
condition_pairs.append((-9,'m')) # only want -9dB for 1 distractor
condition_pairs.append((-9,'f')) # only want -9dB for 1 distractor


print(condition_pairs)
# condition_pairs.append((-9,1)) # only want -9dB for 1 distractor

# file_out_dir = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes_mturk/')

[(-6, 'm'), (-6, 'f'), (-6, ('m', 'm')), (-6, ('m', 'f')), (-6, ('f', 'f')), (-6, ('m', 'm', 'm', 'm')), (-6, ('m', 'm', 'm', 'f')), (-6, ('m', 'm', 'f', 'f')), (-6, ('m', 'f', 'f', 'f')), (-6, ('f', 'f', 'f', 'f')), (-6, 'ssn'), (-3, 'm'), (-3, 'f'), (-3, ('m', 'm')), (-3, ('m', 'f')), (-3, ('f', 'f')), (-3, ('m', 'm', 'm', 'm')), (-3, ('m', 'm', 'm', 'f')), (-3, ('m', 'm', 'f', 'f')), (-3, ('m', 'f', 'f', 'f')), (-3, ('f', 'f', 'f', 'f')), (-3, 'ssn'), (0, 'm'), (0, 'f'), (0, ('m', 'm')), (0, ('m', 'f')), (0, ('f', 'f')), (0, ('m', 'm', 'm', 'm')), (0, ('m', 'm', 'm', 'f')), (0, ('m', 'm', 'f', 'f')), (0, ('m', 'f', 'f', 'f')), (0, ('f', 'f', 'f', 'f')), (0, 'ssn'), (3, 'm'), (3, 'f'), (3, ('m', 'm')), (3, ('m', 'f')), (3, ('f', 'f')), (3, ('m', 'm', 'm', 'm')), (3, ('m', 'm', 'm', 'f')), (3, ('m', 'm', 'f', 'f')), (3, ('m', 'f', 'f', 'f')), (3, ('f', 'f', 'f', 'f')), (3, 'ssn'), (-9, 'm'), (-9, 'f')]


In [ ]:
for condition in condition_pairs:
    snr, distractor = condition
    n_distractors = len(distractor)

    all_conditions.append({'snr':snr,
                          'n_distractors':n_distractors,
                          'distractor':distractor})

    # shuffle condition order inplace
    ### TO DO make df with each possible combination condition for each target  con
    
    
    
    
    # sample our talkers for one set 
    set_targets = target_timit.groupby('speaker_sex').sample(n=n_female).copy() # replace=False is default
    set_targets.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    set_targets.index = set_targets.index.set_names(['_original_timit_index']) # add original index mapping
    set_targets = set_targets.reset_index() # Reset for 0:399 ixs 
    
    # get set of distractors given targets
    bg_ixs = target_timit[["speaker", 'sentence_id','word']].merge(set_targets[['speaker', 'sentence_id','word']],
                                    how = 'outer' ,indicator=True).loc[lambda x : x['_merge']=='left_only'].index
    bg_timit = target_timit.iloc[bg_ixs]
    bg_timit.index = bg_timit.index.set_names(['_original_timit_index']) # keep original ixs 
    bg_timit = bg_timit.reset_index()
    
    print(f"{len(set_targets.word.unique())} unique words in dataset {set_ix}")
    print(f"{set_targets.word.value_counts().max()} max frequency in dataset {set_ix}")
    
    # Sample cues 
    ## To sample cues, get list of talkers in target set 

    talkers = set_targets.speaker.unique()

    # sample 1 cue per excerpt from viable cues 
    samples_per_talker = {talker:count for talker,count in set_targets.speaker.value_counts().items()}
    viables_cues = cue_timit[cue_timit.speaker.isin(talkers)]

    cues = viables_cues.groupby('speaker').apply(lambda group: group.sample(samples_per_talker[group.name]))
    cues.drop(columns='speaker', inplace=True)
    cues = cues.reset_index()
    cues.rename(columns={'level_1':'_original_cue_timit_index'}, inplace=True)
    
    set_targets.sort_values(by='speaker', inplace=True)
    set_targets.reset_index(inplace=True, drop=True)
    
    ### Merge cues with foregrounds  
    cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = cues[['signal', 'word', '_original_cue_timit_index', 'speaker']]
    # Combine as experiment dataframe
    exp_df = set_targets.join(cues[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']])
    assert (exp_df['speaker'] == exp_df['cue_speaker']).all(), "Cue and Target talkers don't match!"
    
    # Generate trial by trial stim    
    trial_data = {'mixture_signal':[],
            'distractor_signal':[],
            '_original_distractor_timit_indices':[],
            'distractor_words':[],
            'distractor_speakers':[],
            'distractor_conditions':[],
            'distractor_sex':[],
            'snrs':[]}
    cue_signals = []
    cue_snrs = []

    # Loop over trials to make mixtures 
    for i in tqdm(range(n_trials), leave=False):
        # get signal
        signal = exp_df['signal'][i]
        speaker = exp_df['speaker'][i]
        valid_distractors = bg_timit[bg_timit['speaker'] != speaker].reset_index(drop=True) # reset so ixing works
        # get condition 
        snr, condition, distractors = all_conditions[i].values()
        # Get data based on condition
        if condition == 'ssn':
            noise = generate_speech_shaped_noise()
            indices = 'ssn'
            words = 'ssn'
            speakers = 'ssn'

        else:
            if condition == 1:
                distractor_ixs = np.where(valid_distractors.speaker_sex==distractors)[0]
                distractor_ixs = np.random.choice(distractor_ixs, size=1)
                noise = valid_distractors['signal'][distractor_ixs[0]]

            elif condition in [2, 4]:
                talker_sex_counts = dict(zip(*np.unique(distractors, return_counts = True)))
                m_distractors=[]
                f_distractors=[]
                if 'm' in talker_sex_counts:
                    m_distractors = np.where(valid_distractors.speaker_sex =='m')[0]
                    m_distractors = np.random.choice(m_distractors, size=talker_sex_counts['m'])
                if 'f' in talker_sex_counts:
                    f_distractors = np.where(valid_distractors.speaker_sex=='f')[0]
                    f_distractors = np.random.choice(f_distractors, size=talker_sex_counts['f'])
                distractor_ixs = np.concatenate([f_distractors, m_distractors])

                noise = np.stack(valid_distractors['signal'][distractor_ixs]).sum(0) # sum unique distractors 
            # get common feats 
            words = valid_distractors['word'][distractor_ixs].values
            indices = valid_distractors['_original_timit_index'][distractor_ixs].values
            speakers = valid_distractors['speaker'][distractor_ixs].values
        
        # create trial stim 
        noise,_ = rms_normalize(noise)
        signal, _ = rms_normalize(signal)
        mixture, _ = combine_with_noise(signal, noise, snr) # first_scale_factor unused
        mixture, mixture_scale_factor = rms_normalize(mixture)
        cue, _ = rms_normalize(exp_df['cue_signal'][i])
        # rove cue
        cue_signal = cue * mixture_scale_factor
       
        cue_snrs.append(snr)
        cue_signals.append(cue_signal)

        # save values for tiral 
        trial_data['mixture_signal'].append(mixture)
        trial_data['distractor_signal'].append(noise)
        trial_data['_original_distractor_timit_indices'].append(indices)
        trial_data['distractor_words'].append(words)
        trial_data['distractor_speakers'].append(speakers)
        trial_data['distractor_conditions'].append(condition)

        trial_data['distractor_sex'].append(''.join(str(token) for token in distractors))
        trial_data['snrs'].append(snr)
    # convert to trial df 
    trial_data = pd.DataFrame.from_dict(trial_data) 
    # merge with exp_df 
    exp_df = exp_df.join(trial_data)
    # update cues with roved level
    exp_df['cue_snr'] = cue_snrs
    exp_df['cue_signal'] = cue_signals
    
    # get catch trials
    catch_trials = bg_timit.sample(n=n_catch_trials)
    catch_trials.drop(columns=['_full_dataset_index', 'data_split'], inplace=True)

    catch_trials.sort_values(by='speaker', inplace=True)
    catch_trials.reset_index(inplace=True, drop=True)
    
    #  catch cues are just the target signal
    catch_trials[['cue_signal', 'cue_word', '_original_cue_timit_index', 'cue_speaker']] = catch_trials[['signal', 'word', '_original_timit_index', 'speaker']]
    
    catch_trials['mixture_signal'] = catch_trials['signal']
    catch_trials['distractor_signal'] = 'catch trial'
    catch_trials['_original_distractor_timit_indices'] = 'catch trial'
    catch_trials['distractor_words'] = 'catch trial'
    catch_trials['distractor_speakers']  = 'catch trial'
    catch_trials['distractor_conditions'] = 'catch trial'
    catch_trials['snrs'] = 'catch trial'
    
    exp_df = pd.concat([exp_df, catch_trials],ignore_index=True)
    if not file_out_dir.exists():
        file_out_dir.mkdir(parents=True, exist_ok=True)
    ## Save exp_df 
#     df_name = file_out_dir / f"attn_task_dataset_{set_ix:02d}.pdpkl"
#     break
#     exp_df.to_pickle(df_name)
    

In [17]:
word_and_speaker_encodings = pickle.load( open( "/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/word_and_speaker_encodings_jsinv3.pckl", "rb" )) 
class_map = word_and_speaker_encodings['word_idx_to_word']
word_to_ix_map = {val:key for key,val in class_map.items()}



In [19]:
def get_ix_from_words(words):
    if isinstance(words, str):
        return words
    else:
        return [word_to_ix_map[word] for word in words]

target_timit['distractor_word_ints'] = target_timit['distractor_words'].apply(get_ix_from_words)


KeyError: 'distractor_words'

In [9]:
model_stim_df.columns

Index(['_original_timit_index', 'word', 'source', 'speaker', 'sr',
       'signal_length', 'speaker_sex', 'sentence_type', 'sentence_id',
       'dialect_region', 'signal', 'word_int', 'cue_signal', 'cue_word',
       '_original_cue_timit_index', 'cue_speaker', 'mixture_signal',
       'distractor_signal', '_original_distractor_timit_indices',
       'distractor_words', 'distractor_speakers', 'distractor_conditions',
       'distractor_sex', 'snrs', 'cue_snr', 'dataset', 'distractor_word_ints'],
      dtype='object')

In [10]:
out_path = Path('/om2/user/imgriff/datasets/timit/attn_task_dataframes/')

model_stim_df.to_pickle(out_path / 'timit_all_attn_stim_for_model_eval_0_1rms.pdpkl')